# Tutorial FluorophoreSystem

### Naming convention
|name|meaning|
|---|---|
|single state|photophysical state|
|joined state|combinations of single states for each fluorophore|
|unique transition|description of each photophysical transition|
|joined transition|change in joined state|
|emission|fluorescent emission|
|event|detected emission|
|resample / deltat|frame integration time, $\tau_{min}$, lag time$_{min}$|

### Handling of multiple possible transitions
Example: The transition between first excited state $S_1$ and ground state $S_0$ can be via fluorescence or via internal/external conversion followed by vibrational relaxation.

Strategy: 
- effected transition rates are added in initialize.construct_transition_matrices
- the rate used in the algorithm is hence dependent on both transitions
- to later discriminate between the two transitions:
    - process.multiple_transitions to create cumulative sum of the effected transitions
    - process.generate_transition_series inserts random number into cumulative sum

In [1]:
#user = r"\SagixOffice"  # HomeOffice
user = r"\vie43sq"  # University
import sys
sys.path.append(rf"C:\Users{user}\OneDrive - Universität Würzburg\GitHub\Photoswitching")

import numpy as np
import pandas as pd
import src.fluorophore_systems as fs
import src.miscellaneous as mi

%load_ext autoreload
%autoreload 2

C:\Users\vie43sq\Miniconda3\envs\MarkovModels\lib\site-packages\pycorrelate\pycorrelate.py:118: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def ucorrelate(t, u, maxlag=None):


In [2]:
transitions = [['S0_S1', 7e6, "excitation", "EXC", False],  
         ['S1_S0', 1e9, "fluorescent emission", "FLU", True], 
         ['S1_T1', 1e6, "intersystem crossing ST", "ISCST", False],   
         ['T1_S0', 5e5, "intersystem crossing TS", "ISCTS", False],
         ['S1_S0', 1e9, "internal conversion S", "ICS", False],
        ['S1_S0__S0_S1', 1e10, "energy transfer", "FRET", False]]

## Initialize object

In [7]:
system = fs.FluorophoreSystem(number_fluorophores=2, distances=1, transitions=transitions)

['S0', 'S0']
S1
['S0', 'S1']
S1
['S0', 'T1']
S1
['S1', 'S0']
S1
['S1', 'S1']
S1
['S1', 'T1']
S1
['T1', 'S0']
S1
['T1', 'S1']
S1
['T1', 'T1']
S1
['S0', 'S0']
S1
['S0', 'S1']
S1
['S0', 'T1']
S1
['S1', 'S0']
S1
['S1', 'S1']
S1
['S1', 'T1']
S1
['T1', 'S0']
S1
['T1', 'S1']
S1
['T1', 'T1']
S1
['S0', 'S0']
S1
['S0', 'S1']
S1
['S0', 'T1']
S1
['S1', 'S0']
S1
['S1', 'S1']
S1
['S1', 'T1']
S1
['T1', 'S0']
S1
['T1', 'S1']
S1
['T1', 'T1']
S1
['S0', 'S0']
S1
['S0', 'S1']
S1
['S0', 'T1']
S1
['S1', 'S0']
S1
['S1', 'S1']
S1
['S1', 'T1']
S1
['T1', 'S0']
S1
['T1', 'S1']
S1
['T1', 'T1']
S1
['S0', 'S0']
S1
['S0', 'S1']
S1
['S0', 'T1']
S1
['S1', 'S0']
S1
['S1', 'S1']
S1
['S1', 'T1']
S1
['T1', 'S0']
S1
['T1', 'S1']
S1
['T1', 'T1']
S1
['S0', 'S0']
S1
['S0', 'S1']
S1
['S0', 'T1']
S1
['S1', 'S0']
S1
['S1', 'S1']
S1
['S1', 'T1']
S1
['T1', 'S0']
S1
['T1', 'S1']
S1
['T1', 'T1']
S1
['S0', 'S0']
S1
['S0', 'S1']
S1
['S0', 'T1']
S1
['S1', 'S0']
S1
['S1', 'S1']
S1
['S1', 'T1']
S1
['T1', 'S0']
S1
['T1', 'S1']
S1
['T1', '

In [4]:
system.single_states

{0: 'S0', 1: 'S1', 2: 'T1'}

In [5]:
system.unique_transitions

,name,rate,trivial_name,abbreviation,fluorescence
id,,,,,
0,S0_S1,7.00e+06,excitation,EXC,False
1,S1_S0,1.00e+09,fluorescent emission,FLU,True
2,S1_T1,1.00e+06,intersystem crossing ST,ISCST,False
3,T1_S0,5.00e+05,intersystem crossing TS,ISCTS,False
4,S1_S0,1.00e+09,internal conversion S,ICS,False
5,S1_S0__S0_S1,1.00e+10,energy transfer,FRET,False


In [6]:
system.joined_states

,name,single_states,single_state_counter,absorbing
id,,,,
0,S0_S0,"[0, 0]","[2, 0, 0]",False
1,S0_S1,"[0, 1]","[1, 1, 0]",False
2,S0_T1,"[0, 2]","[1, 0, 1]",False
3,S1_S0,"[1, 0]","[1, 1, 0]",False
4,S1_S1,"[1, 1]","[0, 2, 0]",False
5,S1_T1,"[1, 2]","[0, 1, 1]",False
6,T1_S0,"[2, 0]","[1, 0, 1]",False
7,T1_S1,"[2, 1]","[0, 1, 1]",False
8,T1_T1,"[2, 2]","[0, 0, 2]",False


In [7]:
system.joined_transitions

,name,joined_states_id,unique_transition_id,rate,trivial_name,fluorescence
id,,,,,,
0,S0_S0__S0_S1,"(0, 1)",0,7.00e+06,excitation,False
1,S0_S0__S1_S0,"(0, 3)",0,7.00e+06,excitation,False
2,S0_S1__S1_S1,"(1, 4)",0,7.00e+06,excitation,False
3,S0_T1__S1_T1,"(2, 5)",0,7.00e+06,excitation,False
4,S1_S0__S1_S1,"(3, 4)",0,7.00e+06,excitation,False
5,T1_S0__T1_S1,"(6, 7)",0,7.00e+06,excitation,False
6,S0_S1__S0_S0,"(1, 0)",1,1.00e+09,fluorescent emission,True
7,S1_S0__S0_S0,"(3, 0)",1,1.00e+09,fluorescent emission,True
8,S1_S1__S0_S1,"(4, 1)",1,1.00e+09,fluorescent emission,True


In [8]:
system.transition_matrix

array([[0.00000000e+00, 5.00000000e-01, 0.00000000e+00, 5.00000000e-01,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [1.66555630e-01, 0.00000000e+00, 8.32778148e-05, 8.32778148e-01,
        5.82944704e-04, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [6.66666667e-02, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 9.33333333e-01, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [1.66555630e-01, 8.32778148e-01, 0.00000000e+00, 0.00000000e+00,
        5.82944704e-04, 0.00000000e+00, 8.32778148e-05, 0.00000000e+00,
        0.00000000e+00],
       [0.00000000e+00, 4.99750125e-01, 0.00000000e+00, 4.99750125e-01,
        0.00000000e+00, 2.49875062e-04, 0.00000000e+00, 2.49875062e-04,
        0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 9.99250562e-01, 2.49812641e-04,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        4.9

In [9]:
system.row_sums

array([1.4000e+07, 1.2008e+10, 7.5000e+06, 1.2008e+10, 4.0020e+09,
       2.0015e+09, 7.5000e+06, 2.0015e+09, 1.0000e+06])

In [10]:
system.graph

In [11]:
system.plot

## Method simulate()

In [12]:
system.simulate(n_steps=500, seed=100)

In [13]:
system.time_series

array([0.00000000e+00, 1.28818253e-08, 1.29852398e-08, 1.48923019e-08,
       1.49119046e-08, 1.49430290e-08, 1.49445858e-08, 1.49831593e-08,
       1.50257119e-08, 1.50260643e-08, 1.50477197e-08, 1.50479136e-08,
       1.51255702e-08, 1.51379196e-08, 1.51686484e-08, 1.52471137e-08,
       4.87619684e-08, 4.88696081e-08, 4.92211687e-08, 4.94398198e-08,
       4.94901303e-08, 4.94938795e-08, 4.95508229e-08, 4.95858114e-08,
       4.97807293e-08, 4.98247305e-08, 4.98676681e-08, 9.12770726e-08,
       9.13222945e-08, 9.13508611e-08, 1.71661272e-07, 1.71685449e-07,
       1.71748982e-07, 2.30050936e-07, 2.30150378e-07, 2.30179296e-07,
       2.30265112e-07, 2.30321151e-07, 2.30413394e-07, 2.30447646e-07,
       2.30493707e-07, 2.30514407e-07, 2.30620920e-07, 2.30647140e-07,
       2.30787839e-07, 2.30810655e-07, 2.30811119e-07, 2.30829121e-07,
       2.30831406e-07, 2.30843904e-07, 2.30947495e-07, 2.31052302e-07,
       2.31286900e-07, 2.31555325e-07, 2.31581929e-07, 2.39779678e-07,
      

In [14]:
system.time_step_series

array([0.00000000e+00, 1.28818253e-08, 1.03414550e-10, 1.90706206e-09,
       1.96027040e-11, 3.11244153e-11, 1.55685219e-12, 3.85734364e-11,
       4.25526187e-11, 3.52418170e-13, 2.16553651e-11, 1.93888659e-13,
       7.76566709e-11, 1.23493053e-11, 3.07288847e-11, 7.84652686e-11,
       3.35148547e-08, 1.07639753e-10, 3.51560602e-10, 2.18651042e-10,
       5.03105047e-11, 3.74917992e-12, 5.69433996e-11, 3.49885807e-11,
       1.94917848e-10, 4.40012227e-11, 4.29376131e-11, 4.14094045e-08,
       4.52219236e-11, 2.85665554e-11, 8.03104110e-08, 2.41767106e-11,
       6.35333691e-11, 5.83019541e-08, 9.94422541e-11, 2.89179855e-11,
       8.58160159e-11, 5.60382377e-11, 9.22432423e-11, 3.42516695e-11,
       4.60617572e-11, 2.06992919e-11, 1.06513161e-10, 2.62196790e-11,
       1.40699656e-10, 2.28159315e-11, 4.63577060e-13, 1.80024989e-11,
       2.28485178e-12, 1.24981036e-11, 1.03590738e-10, 1.04806808e-10,
       2.34598494e-10, 2.68424366e-10, 2.66047703e-11, 8.19774915e-09,
      

In [15]:
system.state_series

array([0, 3, 0, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 0, 1, 3, 1, 3, 1, 3,
       1, 3, 1, 3, 0, 3, 1, 0, 1, 3, 0, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1,
       3, 1, 3, 1, 3, 1, 3, 1, 4, 1, 0, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3,
       1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 0,
       3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 0, 1, 3, 0, 3, 1, 3,
       1, 3, 1, 0, 1, 0, 3, 0, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 0, 3, 1,
       3, 1, 3, 1, 3, 1, 3, 0, 3, 1, 0, 3, 1, 3, 1, 0, 1, 3, 0, 1, 3, 1,
       3, 1, 3, 0, 1, 3, 1, 3, 1, 3, 1, 0, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1,
       3, 1, 3, 1, 3, 1, 3, 1, 0, 3, 1, 3, 1, 3, 1, 3, 0, 3, 1, 3, 1, 3,
       1, 3, 1, 0, 3, 0, 1, 0, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 0, 1, 3, 1,
       3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 0, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3,
       1, 3, 1, 3, 0, 1, 3, 1, 3, 1, 0, 1, 3, 1, 3, 1, 3, 0, 1, 3, 1, 3,
       0, 1, 3, 1, 0, 3, 0, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 1, 0, 3,
       1, 0, 3, 1, 3, 1, 0, 1, 3, 0, 1, 0, 1, 3, 1,

The total time of simulation in minutes:

In [16]:
system.time_series[-1] / 60

6.04154881773402e-08

## Method process()
Analysis of complete simulation.

In [17]:
system.process()

In [18]:
system.single_state_series

array([[0., 1., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [19]:
system.transition_series

array([0, 4, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 0, 5, 5, 5, 5, 5, 5,
       5, 5, 5, 4, 0, 5, 1, 0, 5, 1, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
       5, 5, 5, 5, 5, 5, 5, 0, 4, 4, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
       5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 0,
       5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 0, 5, 1, 0, 5, 5, 5,
       5, 5, 4, 0, 4, 0, 1, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 0, 5, 5,
       5, 5, 5, 5, 5, 5, 1, 0, 5, 1, 0, 5, 5, 5, 4, 0, 5, 4, 0, 5, 5, 5,
       5, 5, 4, 0, 5, 5, 5, 5, 5, 5, 4, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
       5, 5, 5, 5, 5, 5, 5, 4, 0, 5, 5, 5, 5, 5, 5, 4, 0, 5, 5, 5, 5, 5,
       5, 5, 1, 0, 4, 0, 1, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 0, 5, 5, 5,
       5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
       5, 5, 5, 4, 0, 5, 5, 5, 5, 4, 0, 5, 5, 5, 5, 5, 4, 0, 5, 5, 5, 4,
       0, 5, 5, 1, 0, 4, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 0, 5,
       1, 0, 5, 5, 5, 4, 0, 5, 4, 0, 1, 0, 5, 5, 5,

In [20]:
system.single_state_lifetimes

{'single_state_occurrences': [array([0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0.,
         1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
         0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0.,
         1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
         0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0.,
         1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
         0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0.,
         1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
         0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0.,
         1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
         0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0.,
         1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
         0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,

In [21]:
system.transition_lifetimes

{'transition_occurrences': [array([0, 4, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
         5, 4, 0, 5, 5, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
         5, 5, 0, 4, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
         5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 0, 5, 5, 5, 5, 5, 5, 5,
         5, 5, 5, 5, 5, 5, 5, 1, 5, 1, 0, 5, 5, 5, 5, 5, 0, 1, 5, 5, 5, 5,
         5, 5, 5, 5, 5, 5, 0, 5, 5, 5, 5, 5, 5, 5, 5, 1, 0, 5, 0, 5, 5, 5,
         5, 4, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 0, 5, 5, 5, 5, 5, 5, 5,
         5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 0, 5, 5, 5, 5, 5, 5, 4, 0, 5, 5, 5,
         5, 5, 5, 5, 0, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5,
         5, 5, 5, 5, 5, 5, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4,
         5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 4, 5, 5, 0, 4, 5, 5, 5, 5,
         5, 5, 5, 5, 5, 5, 5, 5, 0, 5, 0, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5,
         5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 0, 1, 5, 5, 5, 5, 5, 4, 0, 5, 5, 5,

## Method emitters()
Analysis only using events. 

In [22]:
system.emitters(photon_collection_rate=0.9, resample="5ms", emccd_gain=10)

In [23]:
system.emissions

array([ 29,  32,  87, 103, 106, 117, 129, 139, 142, 201, 205, 230, 268,
       284, 287, 297, 317, 319, 351, 360, 371, 383, 400, 403, 441, 445,
       456, 460, 475, 498], dtype=int64)

In [24]:
system.events

array([ 29,  32,  87, 106, 117, 129, 139, 142, 201, 205, 230, 268, 284,
       287, 297, 317, 319, 351, 360, 371, 400, 403, 445, 456, 460, 475,
       498], dtype=int64)

In [25]:
system.time_points_events

array([9.13508611e-08, 1.71748982e-07, 2.42149692e-07, 3.08784952e-07,
       5.27870238e-07, 5.72886721e-07, 5.81610113e-07, 6.38938393e-07,
       7.93994674e-07, 9.29770405e-07, 1.16560518e-06, 1.26709028e-06,
       1.50196748e-06, 1.65473528e-06, 1.85376170e-06, 2.09254448e-06,
       2.10528686e-06, 2.31817933e-06, 2.44040515e-06, 2.48899698e-06,
       2.54081997e-06, 2.54480111e-06, 2.78683882e-06, 2.92753504e-06,
       3.03678610e-06, 3.44784284e-06, 3.54104116e-06])

In [26]:
system.event_time_series

0.00e+00   2.43e+02
dtype: float64

In [27]:
system.on_periods

array([1])

In [28]:
system.off_periods

array([], dtype=int32)

In [29]:
system.emission_statistics

{'total_emissions': 30,
 'total_events': 27,
 'mean_time_between_events': 1.326803961285903e-07}

## Method fcs()

In [30]:
system.fcs(normalize=True, log=True, m=2, deltat=None)

Logarithmic autocorrelation not possible. Event_time_series too short.


C:\Users\SagixOffice\miniconda3\envs\MarkovModels\lib\site-packages\multipletau\core.py:215: RuntimeWarning: divide by zero encountered in double_scalars
  if np.abs(traceavg) / np.median(np.abs(trace)) < ZERO_CUTOFF:


## Parameters

In [31]:
system.parameter_collection

0
init     number_fluorophores                                                        2
         distances                                                                  1
         single_states                                                   [S0, S1, T1]
         transitions                [[S0_S1, 7000000.0, excitation, EXC, False], [...
simulate n_steps                                                                  500
         seed                                                                     100
process  seed                                                                     100
emitters photon_collection_rate                                              9.00e-01
         resample                                                                 5ms
         emccd_gain                                                                10
         threshold                                                                  0
         memory                                                                     0
         remove_heading_off_period                                              False
         seed                                                                     100
fcs      normalize                                                               True
         log                                                                     True
         m                                                                          2
         deltat                                                                   5ms

## Time complexity

In [32]:
mi.time_complexity(mode='number_fluorophores', number_fluorophores=4, simulation_steps_exp=7, seed=100, transitions=transitions)

[68.8288140296936, 69.26811528205872, 70.83010816574097, 73.81850671768188]

Suppose there is an absorbing photophysical state 'B'. The simulation works with joined states, i.e., the absorbing state with 2 fluorophores would be B_B, with three fluorophores B_B_B and so on. The number of simulation steps needed to reach this state scales linearily, meaning that a system of 4 fluorophores needs on average 2 times as many steps as a system of 2 fluorophores to reach this state. \
The time complexity measurement shows that for a fixed amount of simulation steps, the number of fluorophores $n$ does not influence the time. Hence, if the goal is to reach the absorbing state of a system, the time complexity is $\mathcal{O}(n)$.

In [3]:
mi.time_complexity(mode='simulation_steps', number_fluorophores=2, simulation_steps_exp=8, seed=10, transitions=transitions)

[0.050017356872558594,
 0.10936856269836426,
 0.7730987071990967,
 7.71937370300293,
 77.35079503059387,
 932.5235500335693]

Suppose $n$ is the number of simulation steps, then the time complexity is $\mathcal{O}(n)$. For high n, the time complexity might slightly increase due to memory shortage.

## Subclasses

### Cy5

In [6]:
system = fs.Cy5(number_fluorophores=2, distances=1, user=user, irradiance=2.5, wavelength=640, dstorm_parameters=dict())

In [7]:
system.unique_transitions

,name,rate,trivial_name,abbreviation,fluorescence
id,,,,,
0,S0_S1,7.27e+06,excitation,EXC,False
1,S1_S0,2.70e+08,fluorescent emission,FLU,True
2,S1_T1,8.30e+05,intersystem crossing ST,ISCST,False
3,S1_S0,7.09e+08,internal conversion S,ICS,False
4,S1_Cis,2.00e+07,isomerization,ISO,False
5,Cis_S0,1.37e+05,backisomerization,BISO,False
6,T1_S0,5.00e+05,intersystem crossing TS,ISCTS,False
7,T1_OFF,9.60e+06,reduction,RED,False
8,OFF_S0,1.00e+01,oxidation,OX,False
